In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import spacy
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
nlp = spacy.load("en_core_web_lg")

In [ ]:
text='AGRO SPRAY PUMP WITH 3HP Motor RSP-3'

In [ ]:
category_data=pd.read_excel("full unspsc list.xlsx")

In [ ]:
def clean_text(text):
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\d+', '', text)
    doc=nlp(text)
    lemmatized_tokens=[token.lemma_ for token in doc]
    lemmatized_text = ' '.join(lemmatized_tokens)
    lemmatized_text=lemmatized_text.lower().strip()
    return lemmatized_text

In [ ]:
cleaned_text=clean_text(text)

In [ ]:
cleaned_text

In [ ]:
category_data['Level 2 Cleaned']=category_data['Level 2 Description'].apply(clean_text)
category_data['Level 3 Cleaned']=category_data['Level 3 Description'].apply(clean_text)
category_data['Level 4 Cleaned']=category_data['Level 4 Description'].apply(clean_text)

In [ ]:
model= SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
text_embeddings=model.encode(cleaned_text)

In [ ]:
level_2_embeddings = model.encode(category_data['Level 2 Cleaned'].tolist())
level_3_embeddings = model.encode(category_data['Level 3 Cleaned'].tolist())
level_4_embeddings = model.encode(category_data['Level 4 Cleaned'].tolist())

In [ ]:
similarity_2=model.similarity(text_embeddings,level_2_embeddings)[0]
similarity_3=model.similarity(text_embeddings,level_3_embeddings)[0]
similarity_4=model.similarity(text_embeddings,level_4_embeddings)[0]

In [ ]:
similarity_2_array=np.array(similarity_2)
similarity_3_array=np.array(similarity_3)
similarity_4_array=np.array(similarity_4)

In [ ]:
result_df=pd.DataFrame({
    'Level 2':category_data['Level 2 Cleaned'],
    'Level 2 Similarity':similarity_2_array,
    'Level 3':category_data['Level 3 Cleaned'],
    'Level 3 Similarity':similarity_3_array,
    'Level 4':category_data['Level 4 Cleaned'],
    'Level 4 Similarity':similarity_4_array
})

In [ ]:
result_df['Total']=result_df[['Level 2 Similarity', 'Level 3 Similarity', 'Level 4 Similarity']].sum(axis=1)

In [ ]:
result_df_sorted = result_df.sort_values(by='Total', ascending=False)

In [ ]:
result_df_sorted.head()

In [ ]:
text='IMPELLER P NO.159 PUMP UP 100/38'
cleaned_text=clean_text(text)

text_embeddings=model.encode(cleaned_text)

similarity_2=model.similarity(text_embeddings,level_2_embeddings)[0]
similarity_3=model.similarity(text_embeddings,level_3_embeddings)[0]
similarity_4=model.similarity(text_embeddings,level_4_embeddings)[0]

similarity_2_array=np.array(similarity_2)
similarity_3_array=np.array(similarity_3)
similarity_4_array=np.array(similarity_4)

result_df_2=pd.DataFrame({
    'Level 2':category_data['Level 2 Cleaned'],
    'Level 2 Similarity':similarity_2_array,
    'Level 3':category_data['Level 3 Cleaned'],
    'Level 3 Similarity':similarity_3_array,
    'Level 4':category_data['Level 4 Cleaned'],
    'Level 4 Similarity':similarity_4_array
})

result_df_2['Total']=result_df_2[['Level 2 Similarity', 'Level 3 Similarity', 'Level 4 Similarity']].sum(axis=1)
result_df_2_sorted = result_df_2.sort_values(by='Total', ascending=False)

result_df_2_sorted.head(10)

result_df_2_sorted.to_excel('IMPELLER.xlsx',index=False)

In [ ]:
clean_text('IMPELLER P NO.159 PUMP UP 100/38')

In [ ]:
validation=pd.read_excel('validation_lookup_comparision.xlsx')

In [ ]:
# mapped_categories=[]

# for index,row in validation.iterrows():
#     cleaned_text_text= clean_text(validation['Material description'][index])
#     text_embeddings=model.encode(cleaned_text)
#     similarity_2=model.similarity(text_embeddings,level_2_embeddings)[0]
#     similarity_3=model.similarity(text_embeddings,level_3_embeddings)[0]
#     similarity_4=model.similarity(text_embeddings,level_4_embeddings)[0]
    
    
#     similarity_2 = similarity_2.detach().numpy() if isinstance(similarity_2, torch.Tensor) else similarity_2
#     similarity_3 = similarity_3.detach().numpy() if isinstance(similarity_3, torch.Tensor) else similarity_3
#     similarity_4 = similarity_4.detach().numpy() if isinstance(similarity_4, torch.Tensor) else similarity_4
    
# #     similarity_2_array=np.array(similarity_2)
# #     similarity_3_array=np.array(similarity_3)
# #     similarity_4_array=np.array(similarity_4)
#     total_similarity = similarity_2 + similarity_3 + similarity_4
#     total_similarity_array = np.array([similarity_2, similarity_3, similarity_4])
    
# #     for i in range(len(similarity_2)):
# #         sum_similarity=similarity_2[i]+similarity_3[i]+similarity_4[i]
# #         total.append(sum_similarity)
        
# #     total=np.array(total)
#     top_5_indices = total_similarity_array.argsort()[-5:][::-1]
#     top_5_categories = category_data['Level 4 Description'].iloc[top_5_indices].tolist()
#     mapped_categories.append(", ".join(top_5_categories))
    
    
# validation['Mapped Level 4 Categories']= mapped_categories
    

In [ ]:
top_5_categories = []

for product_name in validation['Material description']:

    cleaned_text = clean_text(product_name)
    
    text_embeddings = model.encode(cleaned_text)
    

    similarity_2 = model.similarity(text_embeddings, level_2_embeddings)[0]
    similarity_3 = model.similarity(text_embeddings, level_3_embeddings)[0]
    similarity_4 = model.similarity(text_embeddings, level_4_embeddings)[0]

    similarity_2_array = np.array(similarity_2)
    similarity_3_array = np.array(similarity_3)
    similarity_4_array = np.array(similarity_4)

    result_df_2 = pd.DataFrame({
        'Level 2': category_data['Level 2 Cleaned'],
        'Level 2 Similarity': similarity_2_array,
        'Level 3': category_data['Level 3 Cleaned'],
        'Level 3 Similarity': similarity_3_array,
        'Level 4': category_data['Level 4 Cleaned'],
        'Level 4 Similarity': similarity_4_array
    })

    result_df_2['Total'] = result_df_2[['Level 2 Similarity', 'Level 3 Similarity', 'Level 4 Similarity']].sum(axis=1)

    result_df_2_sorted = result_df_2.sort_values(by='Total', ascending=False)

    top_5_level_4 = result_df_2_sorted.head(5)['Level 4'].tolist()

    top_5_categories.append(top_5_level_4)


validation['Top 5 Categories Level 4'] = top_5_categories


validation.to_excel('Top_5_Categories_Level_4.xlsx', index=False)

In [ ]:
validation.head()